In [32]:
!pip install pingouin factor_analyzer -q

In [33]:
!pip install --upgrade numpy scipy pingouin factor_analyzer

In [34]:
import pandas as pd, numpy as np, pingouin as pg
from scipy import stats
from factor_analyzer import FactorAnalyzer, calculate_kmo, calculate_bartlett_sphericity
import re

In [35]:
from google.colab import files
up = files.upload()
nama_file = list(up.keys())[0]
print("File terbaca:", nama_file)

Saving Quiz.xlsx to Quiz (2).xlsx
File terbaca: Quiz (2).xlsx


In [36]:
JUMLAH_KOLOM_DEMOGRAFI = 5
df = pd.read_excel( nama_file)
butir = df.iloc[:, JUMLAH_KOLOM_DEMOGRAFI:]
print("Jumlah responden :", len(df))
print("Jumlah butir :", butir.shape[1])
print("Data kosong :", int(butir.isna().sum().sum()))

Jumlah responden : 181
Jumlah butir : 31
Data kosong : 0


In [37]:
skala = {
"Sangat Tidak Setuju (STS)": 1,
"Tidak Setuju (TS)": 2,
"Netral (N)": 3,
"Setuju (S)": 4,
"Sangat Setuju (SS)": 5,
}
num = butir.replace(skala).apply(pd.to_numeric, errors="coerce")
# Harus 0. Jika lebih dari 0, ada jawaban yang tidak terbaca.
print("Sel yang gagal dikonversi:", int(num.isna().sum().sum()))
num.head()

Sel yang gagal dikonversi: 195


,Jumlah Platform AI yang berlangganan secara premium (bukan free),Penggunaan AI pada tugas atau projek kuliah [Saya menggunakan platform AI hampir setiap kali mendapatkan tugas akademik.],Penggunaan AI pada tugas atau projek kuliah [Saya menggunakan AI untuk mencari solusi atas permasalahan atau soal perkuliahan.],Penggunaan AI pada tugas atau projek kuliah [Saya menggunakan AI untuk mencari ide atau inspirasi dalam mengerjakan tugas./projek],Penggunaan AI pada tugas atau projek kuliah [Saya merasa kesulitan memulai mengerjakan tugas tanpa bantuan AI.],Penggunaan AI pada tugas atau projek kuliah [Saya merasa kurang yakin dengan hasil tugas apabila tidak menggunakan AI.],Penggunaan AI pada tugas atau projek kuliah [Saya percaya bahwa AI dapat memberikan informasi yang membantu penyelesaian tugas.],Penggunaan AI pada tugas atau projek kuliah [Saya memeriksa kembali kebenaran informasi yang diberikan AI sebelum menggunakannya.],Aspek Psikologis penggunaan AI pada tugas atau projek kuliah [Saya merasa perlu menggunakan AI setiap kali mendapatkan tugas atau proyek kuliah.],Aspek Psikologis penggunaan AI pada tugas atau projek kuliah [aya merasa kesulitan menyelesaikan tugas kuliah apabila tidak menggunakan bantuan AI.],...,Kebiasaan penggunaan teknologi digital [Saya jarang memeriksa kembali hasil dari AI karena sudah percaya terhadap ketepatan jawabannya.],Prokrastinasi akademik. [Saya cenderung menunggu hingga mendekati deadline untuk mulai mengerjakan tugas.],Prokrastinasi akademik. [Saya sering menunda pengerjaan tugas karena merasa dapat dikerjakan dengan mudah dan cepat dengan menggunakan AI],Prokrastinasi akademik. [Saya sering menyelesaikan tugas kuliah dalam waktu yang sangat dekat dengan batas pengumpulan.],Prokrastinasi akademik. [Saya sering mengerjakan beberapa tugas sekaligus karena sebelumnya menunda pengerjaannya.],Rsspon emosional [Saya merasa tertekan ketika harus menyelesaikan tugas dalam waktu singkat akibat menunda pengerjaannya.],Rsspon emosional [Saya merasa panik ketika beberapa tugas yang saya tunda memiliki batas waktu pengumpulan yang berdekatan.],Rsspon emosional [Saya merasa kurang puas terhadap hasil tugas yang dikerjakan karena terburu-buru mendekati deadline.],Rsspon emosional [Saya merasa lega setelah berhasil mengumpulkan tugas meskipun pengerjaannya dilakukan AI saat mendekati deadline.],Rsspon emosional [Saya merasa menyesal karena hasil tugas mungkin bisa lebih baik apabila dikerjakan lebih awal.]
0,2.0,3,4,4,4,3,3,4,3,4,...,3,3,3,3,4,4,3,4,4,3
1,1.0,4,4,4,2,3,4,5,3,2,...,2,3,3,3,4,4,4,4,3,4
2,1.0,3,4,5,3,4,4,5,4,4,...,2,3,3,3,3,4,4,3,4,4
3,1.0,5,5,5,5,4,5,5,5,4,...,5,5,4,4,5,4,5,4,5,5
4,1.0,5,4,4,3,4,4,4,3,4,...,2,4,4,3,4,3,4,5,4,5


In [38]:
dimensi = {}
tidak_cocok = []
for kolom in butir.columns:
    m = re.match(r"^\s*(.+?)\s+\[", kolom)
    if m is None:
        tidak_cocok.append(kolom)
        continue
    dimensi.setdefault(m.group(1).strip(), []).append(kolom)

print("Tidak cocok:", tidak_cocok)
for nama, daftar in dimensi.items():
    print(f"{nama}: {len(daftar)} butir")

# Kalau Tidak cocok berisi kolom Durasi/premium, berarti butir belum difilter — ulangi dulu:
butir = df.iloc[:, 6:].drop(columns=[df.columns[20]])

Tidak cocok: ['Jumlah Platform AI yang berlangganan secara premium (bukan free)', 'Durasi (jam) rata-rata penggunaan gadget perhari, untuk , kebutuhan harian sampai dengan hiburan ']
Penggunaan AI pada tugas atau projek kuliah: 7 butir
Aspek Psikologis penggunaan AI pada tugas atau projek kuliah: 7 butir
Kebiasaan penggunaan teknologi digital: 6 butir
Prokrastinasi akademik.: 4 butir
Rsspon emosional: 5 butir


In [39]:
n = len(num)
t = stats.t.ppf(0.975, n - 2)
r_tabel = t / np.sqrt((n - 2) + t**2)
print(f"n = {n} | r tabel (α=0,05; df={n-2}) = {r_tabel:.3f}")
hasil_validitas = []
for nama, daftar in dimensi.items():
    x = num[daftar]
    total = x.sum(axis=1)
    for i, kolom in enumerate(daftar, 1):
        r, p = stats.pearsonr(x[kolom], total)
        r_koreksi, _ = stats.pearsonr(x[kolom], total - x[kolom])
        hasil_validitas.append({
            "Dimensi": nama,
            "Butir": f"B{i}",
            "r hitung": round(r, 3),
            "Corrected r": round(r_koreksi, 3),
            "p-value": round(p, 4),
            "Keterangan": "Valid" if r > r_tabel else "Tidak valid",
        })

n = 181 | r tabel (α=0,05; df=179) = 0.146


In [40]:
tabel_validitas = pd.DataFrame(hasil_validitas)
tabel_validitas

,Dimensi,Butir,r hitung,Corrected r,p-value,Keterangan
0,Penggunaan AI pada tugas atau projek kuliah,B1,0.697,0.557,0.0,Valid
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.582,0.432,0.0,Valid
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.675,0.491,0.0,Valid
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.582,0.377,0.0,Valid
4,Penggunaan AI pada tugas atau projek kuliah,B5,0.578,0.333,0.0,Valid
5,Penggunaan AI pada tugas atau projek kuliah,B6,0.585,0.373,0.0,Valid
6,Penggunaan AI pada tugas atau projek kuliah,B7,0.435,0.226,0.0,Valid
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,0.692,0.524,0.0,Valid
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,0.663,0.489,0.0,Valid
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,0.593,0.386,0.0,Valid


In [41]:
def cronbach_alpha(x):
    k = x.shape[1]
    return k / (k - 1) * (1 - x.var(ddof=1).sum() / x.sum(axis=1).var(ddof=1))

def kategori(a):
    return ("Sangat baik" if a >= 0.90 else "Baik" if a >= 0.80
            else "Reliabel" if a >= 0.70 else "Tidak reliabel")

hasil_reliabilitas = []
alpha_dihapus = []

for nama, daftar in dimensi.items():
    x = num[daftar]
    a = cronbach_alpha(x)
    hasil_reliabilitas.append({"Dimensi": nama, "Jumlah butir": len(daftar),
                               "Cronbach's alpha": round(a, 3), "Kategori": kategori(a)})

    for i, kolom in enumerate(daftar, 1):
        if len(daftar) > 2:
            alpha_dihapus.append({"Dimensi": nama, "Butir": f"B{i}",
                                  "Alpha jika dihapus": round(cronbach_alpha(x.drop(columns=kolom)), 3)})

tabel_reliabilitas = pd.DataFrame(hasil_reliabilitas)
tabel_alpha_dihapus = pd.DataFrame(alpha_dihapus)
display(tabel_reliabilitas)
display(tabel_alpha_dihapus)

,Dimensi,Jumlah butir,Cronbach's alpha,Kategori
0,Penggunaan AI pada tugas atau projek kuliah,7,0.682,Tidak reliabel
1,Aspek Psikologis penggunaan AI pada tugas atau...,7,0.683,Tidak reliabel
2,Kebiasaan penggunaan teknologi digital,6,0.626,Tidak reliabel
3,Prokrastinasi akademik.,4,0.871,Baik
4,Rsspon emosional,5,0.642,Tidak reliabel


,Dimensi,Butir,Alpha jika dihapus
0,Penggunaan AI pada tugas atau projek kuliah,B1,0.607
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.642
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.617
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.651
4,Penggunaan AI pada tugas atau projek kuliah,B5,0.669
5,Penggunaan AI pada tugas atau projek kuliah,B6,0.653
6,Penggunaan AI pada tugas atau projek kuliah,B7,0.688
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,0.610
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,0.621
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,0.650


In [42]:
gabung = tabel_validitas.merge(tabel_alpha_dihapus, on=["Dimensi", "Butir"], how="left")
with pd.ExcelWriter("hasil_uji_validitas_reliabilitas.xlsx") as w:
    gabung.to_excel(w, sheet_name="Validitas", index=False)
    tabel_reliabilitas.to_excel(w, sheet_name="Reliabilitas", index=False)
files.download("hasil_uji_validitas_reliabilitas.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>